# 07 · Distillation

Companion notebook for [Chapter 8](../chapters/ch08-distillation.md).

**You will:** load and verify teacher traces, format them for the student, and
train briefly with plain cross-entropy.

!!! warning "GPU required for the training cell"

    Loading and formatting data works anywhere. Training needs a GPU.

## 0 · Setup

Run this cell first. It picks a device and imports everything the notebook needs.

If you get `ModuleNotFoundError: reasoning_from_scratch`, your kernel is not the
course environment — see the [setup page](../start/setup.md).

In [ ]:
import torch
from reasoning_from_scratch.ch02 import get_device

device = get_device()
print("torch", torch.__version__)

## 1 · The data

12,000 entries from DeepSeek-R1, four fields each. The first thing to do with any
distillation dataset is look at what a trace actually contains.

In [ ]:
from reasoning_from_scratch.ch08 import load_distill_data, format_distilled_answer

data = load_distill_data(partition="deepseek-r1-math-train")
print("entries:", len(data))
print("fields :", list(data[0].keys()))

entry = data[4]
print("\nproblem:", entry["problem"][:200])
print("answer :", entry["gtruth_answer"])
print("\nthinking (first 400 chars):")
print(entry["message_thinking"][:400])
print("\ncontent:")
print(entry["message_content"][:300])

## 2 · Verify before you train

The filter is the whole game. Run the chapter 3 verifier over the teacher's
answers — the published figure is **90.6% (10,871 / 12,000)**.

Start on a subset so you are not waiting on 12,000 generations' worth of parsing.

In [ ]:
from reasoning_from_scratch.ch03 import grade_answer, extract_final_candidate

subset = data[:500]
correct = 0
for e in subset:
    pred = extract_final_candidate(e["message_content"], fallback="number_then_full")
    correct += bool(grade_answer(pred, e["gtruth_answer"]))
print(f"teacher acceptance on {len(subset)}: {correct/len(subset)*100:.1f}%")
print("published figure on all 12,000 : 90.6%")

**Look at a failure.** A teacher that is wrong is a different problem from a teacher
that is right but badly formatted — the first must be dropped, the second can
sometimes be repaired by the verifier's normalisation.

In [ ]:
for e in data[:200]:
    pred = extract_final_candidate(e["message_content"], fallback="number_then_full")
    if not grade_answer(pred, e["gtruth_answer"]):
        print("predicted:", repr(pred))
        print("truth    :", repr(e["gtruth_answer"]))
        print("content  :", e["message_content"][-300:])
        break

## 3 · Format and tokenise

The student must learn `<think>`-then-answer. Note the tokenizer: **base weights,
reasoning tokenizer.**

In [ ]:
from pathlib import Path
from reasoning_from_scratch.ch03 import load_model_and_tokenizer
from reasoning_from_scratch.ch08 import (
    load_reasoning_tokenizer, build_examples, filter_examples_by_max_len,
)

model, _ = load_model_and_tokenizer(which_model="base", device=device, use_compile=False)
tokenizer = load_reasoning_tokenizer()

print(format_distilled_answer(data[4])[:300])

In [ ]:
examples, skipped = build_examples(data, tokenizer)
print("examples:", len(examples), "| skipped:", skipped)

examples = filter_examples_by_max_len(examples, max_len=2048)
lens = [len(e["token_ids"]) for e in examples]
print(f"token lengths: min={min(lens)} median={sorted(lens)[len(lens)//2]} max={max(lens)}")

**Check the prompt boundary.** `prompt_len` is where the loss starts counting. If
this is wrong you train the model to reproduce the question.

In [ ]:
e = examples[0]
print("prompt_len:", e["prompt_len"], "of", len(e["token_ids"]), "tokens")
print("--- prompt ---")
print(tokenizer.decode(e["token_ids"][:e["prompt_len"]]))
print("--- target ---")
print(tokenizer.decode(e["token_ids"][e["prompt_len"]:])[:600])

## 4 · Train

Defaults from the book: `lr=5e-6`, `grad_clip_norm=1.0`, validation on the first
25 examples, per-epoch checkpoints.

In [ ]:
import random
from reasoning_from_scratch.ch08 import train_distillation, evaluate_examples

rng = random.Random(123)
rng.shuffle(examples)
val_examples, train_examples = examples[:25], examples[25:]

print(f"train={len(train_examples)} val={len(val_examples)}")
print(f"val loss before: {evaluate_examples(model, val_examples[:3], device):.3f}")

train_distillation(
    model, train_examples=train_examples[:200], val_examples=val_examples,
    device=device, epochs=1, lr=5e-6, grad_clip_norm=1.0,
    log_every=50, checkpoint_dir="checkpoints",
    csv_log_path="distill_metrics.csv",
)

print(f"val loss after: {evaluate_examples(model, val_examples[:3], device):.3f}")

## 5 · Generate from the student

The test of success: does it emit a `<think>` block before answering?

In [ ]:
from reasoning_from_scratch.ch03 import render_prompt
from reasoning_from_scratch.ch04 import (
    generate_text_stream_concat_flex, generate_text_top_p_stream_cache,
)

raw = "A train travels 120 km in 1.5 hours. What is its average speed in m/s?"
torch.manual_seed(0)
out = generate_text_stream_concat_flex(
    model, tokenizer, render_prompt(raw), device, max_new_tokens=512,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.7, top_p=0.9,
)
print(out)

## 6 · The reference curves

| Epoch | Step | Train loss | Val loss |
| ---: | ---: | ---: | ---: |
| 1 | 50 | 0.766 | 0.699 |
| 1 | 100 | 0.746 | 0.668 |
| 3 | 20,000 | 0.436 | 0.534 |

200 steps will not get you there. What you are checking is the *direction* and that
validation tracks training rather than diverging.

## 7 · Exercises

1. Compare traces from `deepseek-r1-math-train` and `qwen3-235b-a22b-math-train`.
   Which teacher is more verbose? Does filtering remove more of one?
2. Change `max_len` to 512. What fraction is dropped, and what does that tell you
   about the teacher?
3. Train on unfiltered data for 200 steps and compare validation loss. What did
   skipping the verifier cost you?
4. Evaluate your student on MATH-500 with `evaluate_math500_stream`.